# Rozdział 1. Przedstawienie i przygotowanie danych

*American Bankruptcy Dataset — dane finansowe firm z lat 1999–2018*

## 1.1 Cel rozdziału

W tym rozdziale przedstawiono zbiór *American Bankruptcy Dataset* i przygotowano go do dalszej pracy. Zakres obejmuje opis kolumn, wczytanie pliku, sprawdzenie braków i duplikatów oraz uporządkowanie typów danych i kodowania statusu firmy.

## 1.2 Opis zbioru

Każdy wiersz opisuje jedną firmę w jednym roku obrotowym. Zbiór zawiera następujące kolumny:

- `company_name` — identyfikator firmy.
- `fyear` — rok obrotowy.
- `status_label` — status firmy: `alive` lub `failed`.
- `X1`–`X18` — wartości finansowe opisane poniżej.
- `Division`, `MajorGroup` — kody grup działalności.

| Kod | Znaczenie | Kod | Znaczenie |
|---|---|---|---|
| X1 | Aktywa obrotowe | X10 | Aktywa ogółem |
| X2 | Koszt własny sprzedaży | X11 | Dług długoterminowy |
| X3 | Amortyzacja | X12 | EBIT |
| X4 | EBITDA | X13 | Zysk brutto ze sprzedaży |
| X5 | Zapasy | X14 | Zobowiązania krótkoterminowe |
| X6 | Wynik netto | X15 | Zyski zatrzymane |
| X7 | Należności ogółem | X16 | Przychody ogółem |
| X8 | Wartość rynkowa | X17 | Zobowiązania ogółem |
| X9 | Sprzedaż netto | X18 | Koszty operacyjne |

Nazwy X1–X18 przyjęto z dokumentacji wskazanej w notebooku. Opis nie podaje jednostek ani waluty. Status `alive` oznacza brak zgłoszenia upadłości w badanym okresie, a nie potwierdzenie dobrej kondycji finansowej.

## 1.3 Wczytanie danych

Dane wczytano z pliku `dane.xlsx`, z arkusza `american_bankruptcy_dataset (1)`. Do przygotowania zbioru wykorzystano biblioteki pandas i numpy. Poniżej sprawdzono wymiary tabeli oraz wyświetlono pierwsze rekordy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

feature_names = {'X1': 'Aktywa obrotowe', 'X2': 'Koszt własny sprzedaży', 'X3': 'Amortyzacja', 'X4': 'EBITDA', 'X5': 'Zapasy', 'X6': 'Wynik netto', 'X7': 'Należności ogółem', 'X8': 'Wartość rynkowa', 'X9': 'Sprzedaż netto', 'X10': 'Aktywa ogółem', 'X11': 'Dług długoterminowy', 'X12': 'EBIT', 'X13': 'Zysk brutto ze sprzedaży', 'X14': 'Zobowiązania krótkoterminowe', 'X15': 'Zyski zatrzymane', 'X16': 'Przychody ogółem', 'X17': 'Zobowiązania ogółem', 'X18': 'Koszty operacyjne'}
pd.set_option('display.max_columns', 25)
print('pandas:', pd.__version__)
path = Path('dane.xlsx')
if not path.exists():
    path = Path('upload/dane.xlsx')
raw = pd.read_excel(path, sheet_name='american_bankruptcy_dataset (1)')
print('Wymiary:', raw.shape)
print(raw.head().to_string(index=False))

pandas: 2.2.3
Wymiary: (78682, 23)
company_name  fyear status_label       X1         X2       X3       X4       X5       X6        X7        X8         X9      X10       X11        X12        X13         X14      X15      X16         X17      X18 Division  MajorGroup
         C_1 1999.0        alive 511267.0 740998.000 833107.0 180447.0 18373.00 70658.00 89031.000 191226.00    336.018 163816.0    35.163 201026.000    128.348 1024333.000 372.7519 401483.0 1024333.000 935302.0        D          37
         C_1 2000.0        alive 485856.0    701.854 713811.0 179987.0 18577.00    45.79 64367.000 160444.00    320.590 125392.0 18531.000 204065.000 115187.000  874255.000 377.1180 361642.0  874255.000 809888.0        D          37
         C_1 2001.0        alive 436656.0 710199.000 526477.0 217699.0 22496.00  4711.00 27207.000 112244.00 286588.000 150464.0   -58.939    139.603  77528.000  638721.000 364.5928 399964.0  638721.000 611514.0        D          37
         C_1 2002.0        alive 

Wczytana tabela zawiera **78 682 wiersze i 23 kolumny**. Każdy rekord odpowiada jednej firmie w danym roku. Kolumna `fyear` ma początkowo typ zmiennoprzecinkowy, dlatego w dalszym kroku zostanie zmieniona na typ całkowity.

## 1.4 Sprawdzenie kompletności i poprawności zapisu

Sprawdzono braki danych, pełne duplikaty i powtórzenia par firma–rok. Rok oraz kolumny X1–X18 przekonwertowano na wartości liczbowe. Wpisy, których nie można odczytać jako liczby, oraz wartości nieskończone w kolumnach finansowych są oznaczane jako braki.

Sprawdzono także, czy `status_label` zawiera wyłącznie `alive` i `failed` oraz czy status jest stały w obrębie firmy. Na potrzeby dalszej pracy dodano kolumnę `target`: 0 dla `alive` i 1 dla `failed`.

In [ ]:
features = [f'X{i}' for i in range(1, 19)]
df = raw.copy()
for c in ['fyear'] + features:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df[features] = df[features].replace([np.inf, -np.inf], np.nan)
print('Braki po konwersji:', int(df.isna().sum().sum()))
print('Pełne duplikaty:', int(df.duplicated().sum()))
print('Powtórzone pary firma–rok:', int(df.duplicated(['company_name', 'fyear']).sum()))
print('Liczba różnych etykiet w jednej firmie:')
print(df.groupby('company_name').status_label.nunique().value_counts().to_string())
assert set(df.status_label.unique()) == {'alive', 'failed'}
assert not df.duplicated(['company_name', 'fyear']).any()
assert df.groupby('company_name').status_label.nunique().eq(1).all()
df['target'] = df.status_label.map({'alive': 0, 'failed': 1})

Braki po konwersji: 0
Pełne duplikaty: 0
Powtórzone pary firma–rok: 0
Liczba różnych etykiet w jednej firmie:
status_label
1    8971


Po konwersji nie wykryto braków ani pełnych duplikatów. Pary firma–rok są unikalne, więc nie było potrzeby uzupełniania wartości ani usuwania powtórzeń.

Każda firma ma ten sam status we wszystkich swoich rekordach. Etykieta opisuje więc firmę i nie wskazuje daty upadłości. Zachowano jej oryginalne wartości.

## 1.5 Uporządkowanie typów danych

Przed zmianą typu `fyear` sprawdzono, czy kolumna nie zawiera braków ani części ułamkowych. Następnie rok zapisano jako liczbę całkowitą. Kody `Division` i `MajorGroup` pozostają oznaczeniami grup działalności; ich znaczenie opisano w podrozdziale 1.2.

Po dodaniu `target` tabela zawiera 24 kolumny. Pozostałe wartości oraz liczba wierszy nie zostały zmienione.

In [ ]:
assert df['fyear'].notna().all() and (df['fyear'] % 1 == 0).all()
df['fyear'] = df['fyear'].astype(int)
print('Zbiór po przygotowaniu:', df.shape, '(23 kolumny źródłowe + kolumna target)')
print(df.dtypes.value_counts().to_string())

## 1.6 Podsumowanie przygotowania

Wczytano zbiór, sprawdzono jego kompletność i unikalność rekordów oraz uporządkowano zapis roku. Dodano też liczbowe kodowanie statusu firmy. Nie było potrzeby uzupełniania braków ani usuwania duplikatów.

Tak przygotowana tabela stanowi punkt wyjścia do dalszej analizy. Kontrola techniczna nie potwierdza jeszcze poprawności ekonomicznej wartości finansowych; ich zakresy i spójność wymagają osobnego sprawdzenia.
